# Module 5.3: 领域适应与持续学习

## 1. 本章概览

### 📚 学习目标

1. **领域偏移**：理解源域和目标域的分布差异，学会检测领域偏移
2. **领域适应**：掌握 DAPT 和 TAPT 技术的完整实现
3. **灾难性遗忘**：理解并解决持续学习中的遗忘问题
4. **持续学习**：实现 EWC 和 Experience Replay 等防遗忘策略
5. **多领域学习**：实现同时处理多个领域的模型架构

### 🎯 核心问题

- 为什么模型在新领域表现下降？
- 如何有效地适应新领域？
- 如何在学习新任务时保留旧知识？
- 如何设计能处理多个领域的模型？

### 📊 本章内容

- **6 个完整微实践**：从领域偏移检测到多领域学习
- **5 个精美可视化**：词汇分布、训练曲线、遗忘曲线、Fisher 热图、性能雷达图
- **1 个完整案例**：医学文本领域适应端到端实现
- **完整可运行代码**：所有实现都是真实可执行的

### ⏱️ 预计学习时间：4-5小时

---

### 🗺️ 学习路线图

```
领域偏移检测 → DAPT/TAPT 实现 → 灾难性遗忘演示 → 
EWC 防遗忘 → Experience Replay → 多领域学习 → 医学案例
```

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

print("✓ Libraries imported")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. 领域偏移 (Domain Shift)

### 2.1 什么是领域偏移？

**定义**：源域和目标域的数据分布不同

$$P_{\text{source}}(X, Y) \neq P_{\text{target}}(X, Y)$$

### 2.2 常见的领域偏移场景

1. **通用 → 专业领域**
   - 新闻文本 → 医学文献
   - 日常对话 → 法律文档
   - 维基百科 → 科学论文

2. **正式 → 非正式**
   - 书面语 → 口语
   - 标准英语 → 社交媒体文本
   - 新闻 → 推特

3. **语言混合**
   - 单语 → 代码切换文本
   - 英语 → 中英混合
   - 标准语 → 方言

### 2.3 领域偏移的影响

- **性能下降**：模型在新领域准确率显著降低（可能下降 10-30%）
- **词汇分布变化**：新领域有特定术语和表达方式
- **语义理解偏差**：相同词汇在不同领域含义不同
- **句法模式差异**：句子结构和长度分布变化

### 2.4 如何检测领域偏移？

1. **词汇分析**：比较词频分布
2. **嵌入空间**：可视化不同领域的表示
3. **性能监控**：在目标域上测试性能
4. **统计测试**：KL 散度、JS 散度等

## 3. 持续预训练 (Continued Pre-training)

### 3.1 领域自适应预训练 (DAPT)

**Domain-Adaptive Pre-Training (DAPT)** 是一种三阶段的训练流程：

$$\text{通用预训练} \xrightarrow{\text{DAPT}} \text{领域预训练} \xrightarrow{\text{微调}} \text{任务模型}$$

**DAPT 步骤**：
1. 加载通用预训练模型（如 BERT、RoBERTa）
2. 在领域特定语料上继续 MLM (Masked Language Modeling) 训练
3. 在下游任务上微调

### 3.2 为什么 DAPT 有效？

- **学习领域词汇**：适应医学、法律等专业术语
- **捕获领域模式**：理解领域特定的语言结构
- **提高任务性能**：在领域任务上表现更好（通常提升 2-5%）
- **数据效率**：减少下游任务所需的标注数据

### 3.3 DAPT vs 从头训练

| 特性 | 从头训练 | DAPT |
|------|---------|------|
| 数据需求 | 数十亿 tokens | 数百万 tokens |
| 训练时间 | 数周 | 数小时到数天 |
| 通用能力 | 可能丢失 | 保留 |
| 领域性能 | 高 | 高 |

### 3.4 DAPT 最佳实践

- **数据量**：至少 1M tokens，推荐 10M+ tokens
- **学习率**：比预训练小（1e-5 到 5e-5）
- **训练轮数**：通常 3-10 epochs
- **批次大小**：根据 GPU 内存调整（16-64）

In [ ]:
# 🔬 Micro Practice 1: 领域偏移检测

"""
目标：可视化源域和目标域的词汇分布差异
预期结果：理解领域偏移的具体表现
"""

# 模拟通用领域和医学领域的文本
general_texts = [
    "The weather is nice today",
    "I went to the store yesterday",
    "She likes to read books",
    "The movie was very entertaining",
    "We had dinner at a restaurant",
    "He plays basketball every weekend",
    "The cat is sleeping on the couch",
    "They are planning a vacation",
]

medical_texts = [
    "The patient presents with acute myocardial infarction",
    "Administer 500mg of acetaminophen for fever",
    "CT scan reveals pulmonary embolism",
    "Diagnosis: Type 2 diabetes mellitus",
    "Prescribe antibiotic therapy for infection",
    "Monitor vital signs and cardiac rhythm",
    "Surgical intervention required for appendicitis",
    "Laboratory results indicate elevated glucose levels",
]

def analyze_domain_shift(texts1: List[str], texts2: List[str], 
                         domain1_name: str, domain2_name: str):
    """分析两个领域的词汇分布差异"""
    
    # Tokenize (简化版)
    def tokenize(texts):
        words = []
        for text in texts:
            words.extend(text.lower().split())
        return words
    
    words1 = tokenize(texts1)
    words2 = tokenize(texts2)
    
    # 计算词频
    freq1 = Counter(words1)
    freq2 = Counter(words2)
    
    # 获取 top words
    top1 = freq1.most_common(15)
    top2 = freq2.most_common(15)
    
    # 计算独特词汇
    vocab1 = set(words1)
    vocab2 = set(words2)
    unique1 = vocab1 - vocab2
    unique2 = vocab2 - vocab1
    overlap = vocab1 & vocab2
    
    # 可视化
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. 词频对比
    words_top1 = [w for w, c in top1]
    counts_top1 = [c for w, c in top1]
    axes[0, 0].barh(words_top1, counts_top1, color='skyblue')
    axes[0, 0].set_xlabel('Frequency')
    axes[0, 0].set_title(f'Top Words in {domain1_name}')
    axes[0, 0].invert_yaxis()
    
    words_top2 = [w for w, c in top2]
    counts_top2 = [c for w, c in top2]
    axes[0, 1].barh(words_top2, counts_top2, color='lightcoral')
    axes[0, 1].set_xlabel('Frequency')
    axes[0, 1].set_title(f'Top Words in {domain2_name}')
    axes[0, 1].invert_yaxis()
    
    # 2. 词汇重叠
    overlap_data = [len(unique1), len(overlap), len(unique2)]
    labels = [f'{domain1_name}\nOnly', 'Shared', f'{domain2_name}\nOnly']
    colors = ['skyblue', 'lightgreen', 'lightcoral']
    axes[1, 0].bar(labels, overlap_data, color=colors)
    axes[1, 0].set_ylabel('Number of Words')
    axes[1, 0].set_title('Vocabulary Overlap')
    
    # 添加数值标签
    for i, v in enumerate(overlap_data):
        axes[1, 0].text(i, v + 0.5, str(v), ha='center', va='bottom')
    
    # 3. 词汇分布统计
    stats_text = f"""
    {domain1_name} Statistics:
    - Total words: {len(words1)}
    - Unique words: {len(vocab1)}
    - Avg word length: {np.mean([len(w) for w in words1]):.2f}
    
    {domain2_name} Statistics:
    - Total words: {len(words2)}
    - Unique words: {len(vocab2)}
    - Avg word length: {np.mean([len(w) for w in words2]):.2f}
    
    Domain Shift Metrics:
    - Vocabulary overlap: {len(overlap) / len(vocab1 | vocab2) * 100:.1f}%
    - {domain1_name} unique: {len(unique1)}
    - {domain2_name} unique: {len(unique2)}
    """
    
    axes[1, 1].text(0.1, 0.5, stats_text, fontsize=10, 
                    verticalalignment='center', family='monospace')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 打印示例独特词汇
    print(f"\n📊 Domain Shift Analysis")
    print("=" * 60)
    print(f"\n{domain1_name} unique words (sample):")
    print(list(unique1)[:10])
    print(f"\n{domain2_name} unique words (sample):")
    print(list(unique2)[:10])
    print(f"\nShared words (sample):")
    print(list(overlap)[:10])
    
    return {
        'vocab1': vocab1,
        'vocab2': vocab2,
        'overlap': overlap,
        'overlap_ratio': len(overlap) / len(vocab1 | vocab2)
    }

# 执行分析
result = analyze_domain_shift(general_texts, medical_texts, 
                               "General", "Medical")

print(f"\n✓ Domain shift detected!")
print(f"  Vocabulary overlap: {result['overlap_ratio']*100:.1f}%")
print(f"  → Significant domain shift exists!")

In [ ]:
# 🔬 Micro Practice 2: DAPT 完整实现

"""
目标：实现完整的领域自适应预训练流程
预期结果：理解 DAPT 如何改进领域性能
"""

# 简化的 Transformer 模型用于演示
class SimpleTransformer(nn.Module):
    """简化的 Transformer 用于 MLM"""
    def __init__(self, vocab_size=1000, d_model=128, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(512, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=512, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.mlm_head = nn.Linear(d_model, vocab_size)
        
    def forward(self, input_ids, attention_mask=None):
        # Embedding + positional encoding
        seq_len = input_ids.size(1)
        x = self.embedding(input_ids) + self.pos_encoding[:seq_len]
        
        # Transformer encoding
        x = self.transformer(x, src_key_padding_mask=~attention_mask if attention_mask is not None else None)
        
        # MLM prediction
        logits = self.mlm_head(x)
        return logits

class DAPTTrainer:
    """完整的领域自适应预训练实现"""
    
    def __init__(self, model, vocab_size=1000):
        self.model = model
        self.vocab_size = vocab_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        
    def prepare_mlm_data(self, texts: List[str], mlm_probability: float = 0.15):
        """准备 MLM 训练数据"""
        # 简化的 tokenization（实际应使用 tokenizer）
        def simple_tokenize(text):
            # 将文本转换为 token ids（模拟）
            words = text.lower().split()
            return [hash(w) % self.vocab_size for w in words]
        
        all_input_ids = []
        all_labels = []
        all_masks = []
        
        for text in texts:
            token_ids = simple_tokenize(text)
            
            # Padding to fixed length
            max_len = 32
            if len(token_ids) < max_len:
                attention_mask = [1] * len(token_ids) + [0] * (max_len - len(token_ids))
                token_ids = token_ids + [0] * (max_len - len(token_ids))
            else:
                token_ids = token_ids[:max_len]
                attention_mask = [1] * max_len
            
            # Create MLM labels
            labels = token_ids.copy()
            masked_indices = np.random.random(len(token_ids)) < mlm_probability
            
            for i, is_masked in enumerate(masked_indices):
                if is_masked and attention_mask[i] == 1:
                    # 80% mask, 10% random, 10% keep
                    rand = np.random.random()
                    if rand < 0.8:
                        token_ids[i] = self.vocab_size - 1  # [MASK] token
                    elif rand < 0.9:
                        token_ids[i] = np.random.randint(0, self.vocab_size - 1)
                else:
                    labels[i] = -100  # Ignore in loss
            
            all_input_ids.append(token_ids)
            all_labels.append(labels)
            all_masks.append(attention_mask)
        
        return {
            'input_ids': torch.tensor(all_input_ids),
            'labels': torch.tensor(all_labels),
            'attention_mask': torch.tensor(all_masks, dtype=torch.bool)
        }
    
    def train(self, domain_corpus: List[str], epochs: int = 3, 
              lr: float = 1e-4, batch_size: int = 8):
        """完整的训练循环"""
        
        # 准备数据
        data = self.prepare_mlm_data(domain_corpus)
        dataset = torch.utils.data.TensorDataset(
            data['input_ids'], 
            data['labels'],
            data['attention_mask']
        )
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        # 优化器
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss(ignore_index=-100)
        
        # 训练历史
        history = {'loss': [], 'epoch_loss': []}
        
        print(f"🚀 Starting DAPT training...")
        print(f"   Corpus size: {len(domain_corpus)} texts")
        print(f"   Epochs: {epochs}")
        print(f"   Learning rate: {lr}")
        print(f"   Batch size: {batch_size}\n")
        
        self.model.train()
        for epoch in range(epochs):
            epoch_loss = 0
            num_batches = 0
            
            for batch_idx, (input_ids, labels, attention_mask) in enumerate(dataloader):
                input_ids = input_ids.to(self.device)
                labels = labels.to(self.device)
                attention_mask = attention_mask.to(self.device)
                
                # Forward pass
                logits = self.model(input_ids, attention_mask)
                
                # Compute loss (only on masked tokens)
                loss = criterion(logits.view(-1, self.vocab_size), labels.view(-1))
                
                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                num_batches += 1
                history['loss'].append(loss.item())
            
            avg_loss = epoch_loss / num_batches
            history['epoch_loss'].append(avg_loss)
            print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")
        
        print(f"\n✓ DAPT training complete!")
        return history

# 创建模型和训练器
print("Creating model...")
model = SimpleTransformer(vocab_size=1000, d_model=128, nhead=4, num_layers=2)
trainer = DAPTTrainer(model, vocab_size=1000)

# 使用医学领域语料训练
print("\n" + "="*60)
print("DAPT Training on Medical Domain")
print("="*60)

history = trainer.train(medical_texts, epochs=5, lr=1e-4, batch_size=4)

# 可视化训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 1. Batch-level loss
axes[0].plot(history['loss'], alpha=0.6, linewidth=0.8)
axes[0].set_xlabel('Batch')
axes[0].set_ylabel('Loss')
axes[0].set_title('DAPT Training Loss (Batch-level)')
axes[0].grid(True, alpha=0.3)

# 2. Epoch-level loss
epochs_range = range(1, len(history['epoch_loss']) + 1)
axes[1].plot(epochs_range, history['epoch_loss'], marker='o', linewidth=2, markersize=8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Average Loss')
axes[1].set_title('DAPT Training Loss (Epoch-level)')
axes[1].grid(True, alpha=0.3)

# 添加数值标签
for i, loss in enumerate(history['epoch_loss']):
    axes[1].text(i+1, loss, f'{loss:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\n📊 Training Summary:")
print(f"   Initial loss: {history['epoch_loss'][0]:.4f}")
print(f"   Final loss: {history['epoch_loss'][-1]:.4f}")
print(f"   Improvement: {(history['epoch_loss'][0] - history['epoch_loss'][-1]) / history['epoch_loss'][0] * 100:.1f}%")

## 4. 任务自适应预训练 (TAPT)

### 4.1 TAPT vs DAPT

**Task-Adaptive Pre-Training (TAPT)** 使用任务相关的未标注数据进行预训练。

| 特性 | DAPT | TAPT |
|------|------|------|
| 数据来源 | 广泛的领域语料 | 任务相关的未标注数据 |
| 数据量 | 大（10M+ tokens） | 小（10K-1M tokens） |
| 范围 | 广泛的领域知识 | 聚焦于特定任务 |
| 效果 | 通用领域提升 | 特定任务提升更大 |
| 训练时间 | 较长 | 较短 |
| 适用场景 | 多个领域任务 | 单一任务优化 |

### 4.2 组合使用：DAPT + TAPT

最佳实践是结合两者：

$$\text{BERT} \xrightarrow{\text{DAPT}} \text{Domain Model} \xrightarrow{\text{TAPT}} \text{Task Model} \xrightarrow{\text{Fine-tune}} \text{Final Model}$$

**性能提升**（相对于仅微调）：
- 仅 DAPT：+2-3%
- 仅 TAPT：+1-2%
- DAPT + TAPT：+3-5%

### 4.3 何时使用哪种方法？

**使用 DAPT**：
- 有大量领域数据可用
- 需要在多个领域任务上部署
- 领域偏移显著

**使用 TAPT**：
- 领域数据有限
- 任务非常特定
- 需要快速适应

**使用 DAPT + TAPT**：
- 追求最佳性能
- 有足够的计算资源
- 领域和任务都很特定

## 5. 灾难性遗忘 (Catastrophic Forgetting)

### 5.1 问题描述

当模型在新任务上训练时，会**忘记**之前学到的知识：

$$\text{Performance}_{\text{Task A}} \downarrow \text{ after training on Task B}$$

这是神经网络的固有问题，因为参数是共享的。

### 5.2 为什么会遗忘？

1. **参数覆盖**：新任务的梯度更新覆盖了旧任务的权重
2. **优化冲突**：不同任务的最优参数可能不同
3. **缺乏保护**：没有机制保护对旧任务重要的参数
4. **分布偏移**：新任务的数据分布与旧任务不同

### 5.3 衡量遗忘

**后向迁移 (Backward Transfer, BWT)**：

$$BWT = \frac{1}{T-1}\sum_{i=1}^{T-1} (R_{T,i} - R_{i,i})$$

其中：
- $R_{i,j}$：在任务 $j$ 训练后在任务 $i$ 上的性能
- $T$：总任务数
- 负值表示遗忘，正值表示正向迁移

**平均准确率 (Average Accuracy)**：

$$ACC = \frac{1}{T}\sum_{i=1}^{T} R_{T,i}$$

### 5.4 遗忘的严重性

在典型的持续学习场景中：
- **无防护**：旧任务性能可能下降 30-50%
- **随机初始化水平**：严重时可能完全遗忘
- **任务数量影响**：任务越多，遗忘越严重

In [ ]:
# 🔬 Micro Practice 3: TAPT vs DAPT 对比

"""
目标：对比 DAPT 和 TAPT 的效果和数据效率
预期结果：理解何时使用哪种方法
"""

# 模拟不同规模的数据
large_domain_corpus = medical_texts * 10  # DAPT: 大量领域数据
small_task_corpus = medical_texts[:3]      # TAPT: 少量任务数据

print("="*60)
print("TAPT vs DAPT Comparison")
print("="*60)

# 创建两个独立的模型
model_dapt = SimpleTransformer(vocab_size=1000, d_model=128, nhead=4, num_layers=2)
model_tapt = SimpleTransformer(vocab_size=1000, d_model=128, nhead=4, num_layers=2)

trainer_dapt = DAPTTrainer(model_dapt, vocab_size=1000)
trainer_tapt = DAPTTrainer(model_tapt, vocab_size=1000)

print("\n1️⃣ DAPT Training (Large domain corpus)")
print(f"   Data size: {len(large_domain_corpus)} texts")
history_dapt = trainer_dapt.train(large_domain_corpus, epochs=3, lr=1e-4, batch_size=8)

print("\n2️⃣ TAPT Training (Small task-specific corpus)")
print(f"   Data size: {len(small_task_corpus)} texts")
history_tapt = trainer_tapt.train(small_task_corpus, epochs=3, lr=1e-4, batch_size=2)

# 对比可视化
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. 训练曲线对比
axes[0].plot(history_dapt['epoch_loss'], marker='o', label='DAPT', linewidth=2)
axes[0].plot(history_tapt['epoch_loss'], marker='s', label='TAPT', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. 数据效率对比
methods = ['DAPT', 'TAPT']
data_sizes = [len(large_domain_corpus), len(small_task_corpus)]
final_losses = [history_dapt['epoch_loss'][-1], history_tapt['epoch_loss'][-1]]

ax2 = axes[1]
x = np.arange(len(methods))
width = 0.35
bars1 = ax2.bar(x - width/2, data_sizes, width, label='Data Size', color='skyblue')
ax2.set_ylabel('Data Size (texts)', color='skyblue')
ax2.set_xlabel('Method')
ax2.set_title('Data Efficiency Comparison')
ax2.set_xticks(x)
ax2.set_xticklabels(methods)
ax2.tick_params(axis='y', labelcolor='skyblue')

ax2_twin = ax2.twinx()
bars2 = ax2_twin.bar(x + width/2, final_losses, width, label='Final Loss', color='lightcoral')
ax2_twin.set_ylabel('Final Loss', color='lightcoral')
ax2_twin.tick_params(axis='y', labelcolor='lightcoral')

# 添加数值标签
for i, (size, loss) in enumerate(zip(data_sizes, final_losses)):
    ax2.text(i - width/2, size, str(size), ha='center', va='bottom', fontsize=9)
    ax2_twin.text(i + width/2, loss, f'{loss:.3f}', ha='center', va='bottom', fontsize=9)

# 3. 特性对比雷达图
categories = ['Data\nSize', 'Training\nTime', 'Domain\nCoverage', 
              'Task\nSpecificity', 'Generalization']
dapt_scores = [5, 4, 5, 2, 4]  # DAPT 特性评分
tapt_scores = [2, 5, 2, 5, 3]  # TAPT 特性评分

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
dapt_scores += dapt_scores[:1]
tapt_scores += tapt_scores[:1]
angles += angles[:1]

ax3 = plt.subplot(133, projection='polar')
ax3.plot(angles, dapt_scores, 'o-', linewidth=2, label='DAPT', color='skyblue')
ax3.fill(angles, dapt_scores, alpha=0.25, color='skyblue')
ax3.plot(angles, tapt_scores, 's-', linewidth=2, label='TAPT', color='lightcoral')
ax3.fill(angles, tapt_scores, alpha=0.25, color='lightcoral')
ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(categories, size=9)
ax3.set_ylim(0, 5)
ax3.set_title('Method Characteristics', pad=20)
ax3.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax3.grid(True)

plt.tight_layout()
plt.show()

# 打印对比总结
print("\n" + "="*60)
print("📊 Comparison Summary")
print("="*60)
print(f"\nDAPT:")
print(f"  ✓ Data size: {len(large_domain_corpus)} texts")
print(f"  ✓ Final loss: {history_dapt['epoch_loss'][-1]:.4f}")
print(f"  ✓ Best for: Multiple domain tasks, large data available")

print(f"\nTAPT:")
print(f"  ✓ Data size: {len(small_task_corpus)} texts")
print(f"  ✓ Final loss: {history_tapt['epoch_loss'][-1]:.4f}")
print(f"  ✓ Best for: Specific task, limited data")

print(f"\n💡 Recommendation:")
print(f"  • Use DAPT when you have abundant domain data")
print(f"  • Use TAPT when data is limited but task-specific")
print(f"  • Combine both for optimal performance!")

## 6. 持续学习策略

### 6.1 三大类方法

**1. 正则化方法 (Regularization-based)**
- **EWC (Elastic Weight Consolidation)**：保护重要参数
- **SI (Synaptic Intelligence)**：在线估计参数重要性
- **MAS (Memory Aware Synapses)**：基于输出敏感度

**2. 重放方法 (Replay-based)**
- **Experience Replay**：存储旧任务样本
- **Pseudo-rehearsal**：生成伪样本
- **Generative Replay**：使用生成模型

**3. 架构方法 (Architecture-based)**
- **Progressive Networks**：为每个任务添加新列
- **PackNet**：为每个任务分配参数子集
- **Adapters**：任务特定的适配器模块

### 6.2 EWC (Elastic Weight Consolidation) 原理

EWC 通过添加正则化项来保护对旧任务重要的参数：

$$\mathcal{L}(\theta) = \mathcal{L}_B(\theta) + \frac{\lambda}{2}\sum_i F_i(\theta_i - \theta_{A,i}^*)^2$$

其中：
- $\mathcal{L}_B(\theta)$：新任务的损失
- $F_i$：Fisher 信息矩阵，衡量参数 $i$ 的重要性
- $\theta_{A,i}^*$：旧任务训练后的最优参数
- $\lambda$：正则化强度

**Fisher 信息矩阵**：

$$F_i = \mathbb{E}_{x \sim D_A}\left[\left(\frac{\partial \log p(y|x, \theta)}{\partial \theta_i}\right)^2\right]$$

### 6.3 Experience Replay 原理

保存旧任务的样本，在训练新任务时混合使用：

$$\mathcal{L}(\theta) = \mathcal{L}_{\text{new}}(\theta) + \alpha \mathcal{L}_{\text{replay}}(\theta)$$

**优点**：
- 简单有效
- 不需要修改模型结构
- 可以与其他方法结合

**缺点**：
- 需要存储空间
- 可能涉及隐私问题
- 缓冲区大小有限

### 6.4 方法对比

| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| EWC | 无需存储数据 | 计算 Fisher 开销大 | 隐私敏感场景 |
| Replay | 简单有效 | 需要存储空间 | 数据可保存场景 |
| Adapters | 模块化，易切换 | 增加参数量 | 多任务部署 |

In [ ]:
# 🔬 Micro Practice 4: 灾难性遗忘演示

"""
目标：展示顺序学习两个任务时的遗忘现象
预期结果：理解为什么需要持续学习策略
"""

# 简化的分类模型
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=20, num_classes=2):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def generate_task_data(task_id, n_samples=100, input_dim=10):
    """生成任务特定的数据"""
    np.random.seed(42 + task_id)
    torch.manual_seed(42 + task_id)
    
    # 不同任务有不同的数据分布
    if task_id == 0:
        # Task A: 类别 0 在左侧，类别 1 在右侧
        X = torch.randn(n_samples, input_dim)
        y = (X[:, 0] > 0).long()
    else:
        # Task B: 类别 0 在上方，类别 1 在下方
        X = torch.randn(n_samples, input_dim)
        y = (X[:, 1] > 0).long()
    
    return X, y

def train_on_task(model, X, y, epochs=50, lr=0.01):
    """在单个任务上训练"""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return losses

def evaluate(model, X, y):
    """评估模型性能"""
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).float().mean().item()
    model.train()
    return accuracy

print("="*60)
print("Catastrophic Forgetting Demonstration")
print("="*60)

# 生成两个任务的数据
X_taskA, y_taskA = generate_task_data(0, n_samples=100)
X_taskB, y_taskB = generate_task_data(1, n_samples=100)

# 创建模型
model = SimpleClassifier(input_dim=10, hidden_dim=20, num_classes=2)

# 记录性能
performance_history = {
    'task_A': [],
    'task_B': [],
    'stage': []
}

print("\n📚 Stage 1: Training on Task A")
print("-" * 60)
train_on_task(model, X_taskA, y_taskA, epochs=100, lr=0.01)
acc_A = evaluate(model, X_taskA, y_taskA)
acc_B = evaluate(model, X_taskB, y_taskB)
print(f"After Task A training:")
print(f"  Task A accuracy: {acc_A:.3f}")
print(f"  Task B accuracy: {acc_B:.3f} (not trained yet)")

performance_history['task_A'].append(acc_A)
performance_history['task_B'].append(acc_B)
performance_history['stage'].append('After\nTask A')

print("\n📚 Stage 2: Training on Task B (without protection)")
print("-" * 60)
train_on_task(model, X_taskB, y_taskB, epochs=100, lr=0.01)
acc_A_after = evaluate(model, X_taskA, y_taskA)
acc_B_after = evaluate(model, X_taskB, y_taskB)
print(f"After Task B training:")
print(f"  Task A accuracy: {acc_A_after:.3f} ⚠️ (dropped {(acc_A - acc_A_after)*100:.1f}%)")
print(f"  Task B accuracy: {acc_B_after:.3f}")

performance_history['task_A'].append(acc_A_after)
performance_history['task_B'].append(acc_B_after)
performance_history['stage'].append('After\nTask B')

# 计算遗忘度
forgetting = acc_A - acc_A_after
print(f"\n💥 Catastrophic Forgetting Detected!")
print(f"   Forgetting rate: {forgetting*100:.1f}%")
print(f"   Task A performance dropped from {acc_A:.3f} to {acc_A_after:.3f}")

# 可视化遗忘现象
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. 性能变化
stages = performance_history['stage']
x = np.arange(len(stages))
width = 0.35

bars1 = axes[0].bar(x - width/2, performance_history['task_A'], width, 
                     label='Task A', color='skyblue', alpha=0.8)
bars2 = axes[0].bar(x + width/2, performance_history['task_B'], width, 
                     label='Task B', color='lightcoral', alpha=0.8)

axes[0].set_ylabel('Accuracy')
axes[0].set_xlabel('Training Stage')
axes[0].set_title('Performance on Both Tasks')
axes[0].set_xticks(x)
axes[0].set_xticklabels(stages)
axes[0].legend()
axes[0].set_ylim([0, 1.1])
axes[0].grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# 添加遗忘箭头
axes[0].annotate('', xy=(1 - width/2, acc_A_after), xytext=(0 - width/2, acc_A),
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
axes[0].text(0.5, (acc_A + acc_A_after)/2, f'Forgetting\n-{forgetting*100:.1f}%', 
            ha='center', color='red', fontweight='bold', fontsize=10)

# 2. 遗忘曲线（模拟训练过程）
# 模拟在 Task B 训练期间 Task A 性能的下降
task_b_steps = 100
task_a_perf_during_b = [acc_A * np.exp(-0.03 * i) + acc_A_after * (1 - np.exp(-0.03 * i)) 
                        for i in range(task_b_steps)]

axes[1].plot(task_a_perf_during_b, linewidth=2, color='skyblue')
axes[1].axhline(y=acc_A, color='green', linestyle='--', label='Initial Task A performance')
axes[1].axhline(y=acc_A_after, color='red', linestyle='--', label='Final Task A performance')
axes[1].fill_between(range(task_b_steps), acc_A, acc_A_after, alpha=0.2, color='red')
axes[1].set_xlabel('Training Steps on Task B')
axes[1].set_ylabel('Task A Accuracy')
axes[1].set_title('Catastrophic Forgetting Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. 参数变化示意图
# 可视化参数空间中的移动
theta = np.linspace(0, 2*np.pi, 100)
# Task A 最优参数区域
axes[2].fill(np.cos(theta)*0.3 + 0.5, np.sin(theta)*0.3 + 0.5, 
            alpha=0.3, color='skyblue', label='Task A optimal region')
# Task B 最优参数区域
axes[2].fill(np.cos(theta)*0.3 - 0.5, np.sin(theta)*0.3 - 0.5, 
            alpha=0.3, color='lightcoral', label='Task B optimal region')

# 参数移动轨迹
axes[2].plot([0.5, -0.5], [0.5, -0.5], 'k->', linewidth=2, markersize=10)
axes[2].plot(0.5, 0.5, 'o', color='skyblue', markersize=15, label='After Task A')
axes[2].plot(-0.5, -0.5, 's', color='lightcoral', markersize=15, label='After Task B')

axes[2].set_xlim([-1.2, 1.2])
axes[2].set_ylim([-1.2, 1.2])
axes[2].set_xlabel('Parameter Dimension 1')
axes[2].set_ylabel('Parameter Dimension 2')
axes[2].set_title('Parameter Space Movement')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("📊 Analysis")
print("="*60)
print(f"\n✓ Task A was learned successfully: {acc_A:.3f}")
print(f"✗ Task A was forgotten after Task B: {acc_A_after:.3f}")
print(f"✓ Task B was learned successfully: {acc_B_after:.3f}")
print(f"\n💡 This demonstrates why we need continual learning strategies!")
print(f"   → EWC, Experience Replay, or architectural methods")

In [ ]:
# 🔬 Micro Practice 5: EWC 完整实现与验证

"""
目标：实现完整的 EWC 并验证其防遗忘效果
预期结果：对比有无 EWC 的性能差异
"""

class EWC:
    """完整的 Elastic Weight Consolidation 实现"""
    
    def __init__(self, model, lambda_ewc=5000):
        self.model = model
        self.lambda_ewc = lambda_ewc
        self.fisher_dict = {}
        self.optpar_dict = {}
        
    def compute_fisher(self, X, y, num_samples=None):
        """计算 Fisher 信息矩阵"""
        print("Computing Fisher information matrix...")
        
        # 初始化 Fisher 信息
        for name, param in self.model.named_parameters():
            self.fisher_dict[name] = torch.zeros_like(param)
        
        # 使用部分样本计算（如果数据太多）
        if num_samples is not None and len(X) > num_samples:
            indices = torch.randperm(len(X))[:num_samples]
            X_sample = X[indices]
            y_sample = y[indices]
        else:
            X_sample = X
            y_sample = y
        
        self.model.eval()
        
        # 对每个样本计算梯度的平方
        for i in range(len(X_sample)):
            self.model.zero_grad()
            
            # Forward pass
            output = self.model(X_sample[i:i+1])
            
            # 使用对数概率
            log_prob = F.log_softmax(output, dim=1)[0, y_sample[i]]
            
            # Backward pass
            log_prob.backward()
            
            # 累积梯度的平方
            for name, param in self.model.named_parameters():
                if param.grad is not None:
                    self.fisher_dict[name] += param.grad.pow(2)
        
        # 平均
        for name in self.fisher_dict:
            self.fisher_dict[name] /= len(X_sample)
        
        # 保存最优参数
        for name, param in self.model.named_parameters():
            self.optpar_dict[name] = param.data.clone()
        
        self.model.train()
        print(f"✓ Fisher information computed on {len(X_sample)} samples")
        
    def penalty(self):
        """计算 EWC 惩罚项"""
        loss = 0
        for name, param in self.model.named_parameters():
            if name in self.fisher_dict:
                fisher = self.fisher_dict[name]
                optpar = self.optpar_dict[name]
                loss += (fisher * (param - optpar).pow(2)).sum()
        return self.lambda_ewc / 2 * loss

def train_with_ewc(model, X, y, ewc=None, epochs=50, lr=0.01):
    """使用 EWC 训练"""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    task_losses = []
    ewc_losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # 任务损失
        outputs = model(X)
        task_loss = criterion(outputs, y)
        
        # 总损失
        if ewc is not None:
            ewc_loss = ewc.penalty()
            loss = task_loss + ewc_loss
            ewc_losses.append(ewc_loss.item())
        else:
            loss = task_loss
            ewc_losses.append(0)
        
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        task_losses.append(task_loss.item())
    
    return losses, task_losses, ewc_losses

print("="*60)
print("EWC: Preventing Catastrophic Forgetting")
print("="*60)

# 生成数据
X_taskA, y_taskA = generate_task_data(0, n_samples=100)
X_taskB, y_taskB = generate_task_data(1, n_samples=100)

# 实验 1: 无 EWC（baseline）
print("\n🔴 Experiment 1: Without EWC (Baseline)")
print("-" * 60)
model_baseline = SimpleClassifier(input_dim=10, hidden_dim=20, num_classes=2)

print("Training on Task A...")
train_on_task(model_baseline, X_taskA, y_taskA, epochs=100, lr=0.01)
acc_A_baseline = evaluate(model_baseline, X_taskA, y_taskA)
print(f"Task A accuracy: {acc_A_baseline:.3f}")

print("Training on Task B (without EWC)...")
train_on_task(model_baseline, X_taskB, y_taskB, epochs=100, lr=0.01)
acc_A_after_baseline = evaluate(model_baseline, X_taskA, y_taskA)
acc_B_baseline = evaluate(model_baseline, X_taskB, y_taskB)
print(f"Task A accuracy after Task B: {acc_A_after_baseline:.3f}")
print(f"Task B accuracy: {acc_B_baseline:.3f}")
forgetting_baseline = acc_A_baseline - acc_A_after_baseline

# 实验 2: 使用 EWC
print("\n🟢 Experiment 2: With EWC")
print("-" * 60)
model_ewc = SimpleClassifier(input_dim=10, hidden_dim=20, num_classes=2)

print("Training on Task A...")
train_on_task(model_ewc, X_taskA, y_taskA, epochs=100, lr=0.01)
acc_A_ewc = evaluate(model_ewc, X_taskA, y_taskA)
print(f"Task A accuracy: {acc_A_ewc:.3f}")

print("\nComputing Fisher information for Task A...")
ewc = EWC(model_ewc, lambda_ewc=5000)
ewc.compute_fisher(X_taskA, y_taskA, num_samples=50)

print("\nTraining on Task B (with EWC protection)...")
losses, task_losses, ewc_losses = train_with_ewc(
    model_ewc, X_taskB, y_taskB, ewc=ewc, epochs=100, lr=0.01
)
acc_A_after_ewc = evaluate(model_ewc, X_taskA, y_taskA)
acc_B_ewc = evaluate(model_ewc, X_taskB, y_taskB)
print(f"Task A accuracy after Task B: {acc_A_after_ewc:.3f}")
print(f"Task B accuracy: {acc_B_ewc:.3f}")
forgetting_ewc = acc_A_ewc - acc_A_after_ewc

# 可视化对比
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. 性能对比
ax1 = fig.add_subplot(gs[0, :2])
methods = ['Baseline\n(No EWC)', 'With EWC']
x = np.arange(len(methods))
width = 0.25

task_a_before = [acc_A_baseline, acc_A_ewc]
task_a_after = [acc_A_after_baseline, acc_A_after_ewc]
task_b_after = [acc_B_baseline, acc_B_ewc]

bars1 = ax1.bar(x - width, task_a_before, width, label='Task A (before Task B)', color='skyblue')
bars2 = ax1.bar(x, task_a_after, width, label='Task A (after Task B)', color='lightcoral')
bars3 = ax1.bar(x + width, task_b_after, width, label='Task B', color='lightgreen')

ax1.set_ylabel('Accuracy')
ax1.set_title('Performance Comparison: Baseline vs EWC')
ax1.set_xticks(x)
ax1.set_xticklabels(methods)
ax1.legend()
ax1.set_ylim([0, 1.1])
ax1.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

# 2. 遗忘率对比
ax2 = fig.add_subplot(gs[0, 2])
forgetting_rates = [forgetting_baseline * 100, forgetting_ewc * 100]
colors = ['lightcoral', 'lightgreen']
bars = ax2.bar(methods, forgetting_rates, color=colors, alpha=0.7)
ax2.set_ylabel('Forgetting Rate (%)')
ax2.set_title('Catastrophic Forgetting')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax2.grid(True, alpha=0.3, axis='y')

for bar, rate in zip(bars, forgetting_rates):
    ax2.text(bar.get_x() + bar.get_width()/2., rate,
            f'{rate:.1f}%', ha='center', va='bottom' if rate > 0 else 'top', fontsize=10)

# 3. EWC 训练损失分解
ax3 = fig.add_subplot(gs[1, :])
epochs_range = range(len(task_losses))
ax3.plot(epochs_range, task_losses, label='Task Loss', linewidth=2, color='skyblue')
ax3.plot(epochs_range, ewc_losses, label='EWC Penalty', linewidth=2, color='orange')
ax3.plot(epochs_range, losses, label='Total Loss', linewidth=2, color='purple', linestyle='--')
ax3.set_xlabel('Training Steps on Task B')
ax3.set_ylabel('Loss')
ax3.set_title('EWC Training: Loss Decomposition')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Fisher 信息热图
ax4 = fig.add_subplot(gs[2, 0])
# 可视化第一层的 Fisher 信息
fisher_fc1 = ewc.fisher_dict['fc1.weight'].detach().numpy()
im = ax4.imshow(fisher_fc1, cmap='YlOrRd', aspect='auto')
ax4.set_title('Fisher Information (fc1.weight)')
ax4.set_xlabel('Input Dimension')
ax4.set_ylabel('Hidden Dimension')
plt.colorbar(im, ax=ax4, label='Importance')

# 5. 参数重要性分布
ax5 = fig.add_subplot(gs[2, 1])
all_fisher = torch.cat([f.flatten() for f in ewc.fisher_dict.values()])
ax5.hist(all_fisher.numpy(), bins=50, color='skyblue', alpha=0.7, edgecolor='black')
ax5.set_xlabel('Fisher Information Value')
ax5.set_ylabel('Count')
ax5.set_title('Parameter Importance Distribution')
ax5.set_yscale('log')
ax5.grid(True, alpha=0.3)

# 6. 总结统计
ax6 = fig.add_subplot(gs[2, 2])
summary_text = f"""
EWC Results Summary

Baseline (No Protection):
  Task A: {acc_A_baseline:.3f} → {acc_A_after_baseline:.3f}
  Forgetting: {forgetting_baseline*100:.1f}%
  Task B: {acc_B_baseline:.3f}

With EWC Protection:
  Task A: {acc_A_ewc:.3f} → {acc_A_after_ewc:.3f}
  Forgetting: {forgetting_ewc*100:.1f}%
  Task B: {acc_B_ewc:.3f}

Improvement:
  Forgetting reduced by:
  {(forgetting_baseline - forgetting_ewc)*100:.1f}%
  
  Relative improvement:
  {(1 - forgetting_ewc/forgetting_baseline)*100:.1f}%
"""
ax6.text(0.1, 0.5, summary_text, fontsize=10, 
        verticalalignment='center', family='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
ax6.axis('off')

plt.show()

print("\n" + "="*60)
print("📊 EWC Effectiveness Analysis")
print("="*60)
print(f"\n✓ EWC successfully reduced forgetting!")
print(f"  Baseline forgetting: {forgetting_baseline*100:.1f}%")
print(f"  EWC forgetting: {forgetting_ewc*100:.1f}%")
print(f"  Improvement: {(forgetting_baseline - forgetting_ewc)*100:.1f}%")
print(f"\n💡 Key Insights:")
print(f"  • Fisher information identifies important parameters")
print(f"  • EWC penalty prevents large changes to important parameters")
print(f"  • Trade-off: Task B performance may be slightly lower")

## 7. 多领域学习

### 7.1 同时学习多个领域

**挑战**：
- 不同领域可能冲突（负迁移）
- 需要平衡各领域性能
- 参数共享 vs 领域特定

**解决方案**：
1. **领域对抗训练 (Domain Adversarial Training)**
   - 学习领域不变特征
   - 使用梯度反转层
   
2. **混合专家模型 (Mixture of Experts, MoE)**
   - 每个领域有专门的专家
   - 门控网络选择专家
   
3. **领域特定适配器 (Domain-Specific Adapters)**
   - 共享主干网络
   - 每个领域有独立的适配器

### 7.2 领域对抗训练

学习领域不变特征的架构：

```
Input → Feature Extractor → Task Classifier (predict label)
                ↓
         Domain Classifier (predict domain, with gradient reversal)
```

**目标**：
- 任务分类器：最大化任务性能
- 领域分类器：无法区分领域（特征是领域不变的）

### 7.3 Adapter 架构

每个领域使用独立的适配器：

```
Shared Transformer Layers
    ↓
Domain A Adapter → Task A
Domain B Adapter → Task B
Domain C Adapter → Task C
```

**优点**：
- 模块化，易于添加新领域
- 参数高效（只训练适配器）
- 避免灾难性遗忘

In [ ]:
# 🔬 Micro Practice 6: Experience Replay 实现

"""
目标：实现 Experience Replay 并对比与 EWC 的效果
预期结果：理解基于重放的持续学习方法
"""

class ExperienceReplay:
    """经验回放缓冲区"""
    
    def __init__(self, buffer_size=50):
        self.buffer_size = buffer_size
        self.buffer_X = []
        self.buffer_y = []
        
    def add(self, X, y):
        """添加样本到缓冲区"""
        for i in range(len(X)):
            if len(self.buffer_X) >= self.buffer_size:
                # 随机替换（reservoir sampling）
                idx = np.random.randint(0, len(self.buffer_X))
                self.buffer_X[idx] = X[i]
                self.buffer_y[idx] = y[i]
            else:
                self.buffer_X.append(X[i])
                self.buffer_y.append(y[i])
    
    def sample(self, batch_size):
        """从缓冲区采样"""
        if len(self.buffer_X) == 0:
            return None, None
        
        batch_size = min(batch_size, len(self.buffer_X))
        indices = np.random.choice(len(self.buffer_X), batch_size, replace=False)
        
        X_batch = torch.stack([self.buffer_X[i] for i in indices])
        y_batch = torch.tensor([self.buffer_y[i] for i in indices])
        
        return X_batch, y_batch
    
    def __len__(self):
        return len(self.buffer_X)

def train_with_replay(model, X_new, y_new, replay_buffer, epochs=50, lr=0.01, replay_ratio=0.5):
    """使用 Experience Replay 训练"""
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    new_losses = []
    replay_losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # 新任务损失
        outputs_new = model(X_new)
        loss_new = criterion(outputs_new, y_new)
        
        # 重放损失
        if len(replay_buffer) > 0:
            X_replay, y_replay = replay_buffer.sample(int(len(X_new) * replay_ratio))
            outputs_replay = model(X_replay)
            loss_replay = criterion(outputs_replay, y_replay)
            
            # 总损失
            loss = loss_new + loss_replay
            replay_losses.append(loss_replay.item())
        else:
            loss = loss_new
            replay_losses.append(0)
        
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        new_losses.append(loss_new.item())
    
    return losses, new_losses, replay_losses

print("="*60)
print("Experience Replay: Alternative to EWC")
print("="*60)

# 生成数据
X_taskA, y_taskA = generate_task_data(0, n_samples=100)
X_taskB, y_taskB = generate_task_data(1, n_samples=100)

# 实验: 使用 Experience Replay
print("\n🟡 Experiment: With Experience Replay")
print("-" * 60)
model_replay = SimpleClassifier(input_dim=10, hidden_dim=20, num_classes=2)

print("Training on Task A...")
train_on_task(model_replay, X_taskA, y_taskA, epochs=100, lr=0.01)
acc_A_replay = evaluate(model_replay, X_taskA, y_taskA)
print(f"Task A accuracy: {acc_A_replay:.3f}")

print("\nStoring Task A samples in replay buffer...")
replay_buffer = ExperienceReplay(buffer_size=50)
replay_buffer.add(X_taskA, y_taskA)
print(f"Buffer size: {len(replay_buffer)} samples")

print("\nTraining on Task B (with replay)...")
losses, new_losses, replay_losses = train_with_replay(
    model_replay, X_taskB, y_taskB, replay_buffer, 
    epochs=100, lr=0.01, replay_ratio=0.5
)
acc_A_after_replay = evaluate(model_replay, X_taskA, y_taskA)
acc_B_replay = evaluate(model_replay, X_taskB, y_taskB)
print(f"Task A accuracy after Task B: {acc_A_after_replay:.3f}")
print(f"Task B accuracy: {acc_B_replay:.3f}")
forgetting_replay = acc_A_replay - acc_A_after_replay

# 三方法对比可视化
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. 性能对比
methods = ['Baseline', 'EWC', 'Replay']
x = np.arange(len(methods))
width = 0.25

task_a_before = [acc_A_baseline, acc_A_ewc, acc_A_replay]
task_a_after = [acc_A_after_baseline, acc_A_after_ewc, acc_A_after_replay]
task_b_after = [acc_B_baseline, acc_B_ewc, acc_B_replay]

bars1 = axes[0, 0].bar(x - width, task_a_before, width, label='Task A (before)', color='skyblue')
bars2 = axes[0, 0].bar(x, task_a_after, width, label='Task A (after)', color='lightcoral')
bars3 = axes[0, 0].bar(x + width, task_b_after, width, label='Task B', color='lightgreen')

axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Three Methods Comparison')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(methods)
axes[0, 0].legend()
axes[0, 0].set_ylim([0, 1.1])
axes[0, 0].grid(True, alpha=0.3, axis='y')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.2f}', ha='center', va='bottom', fontsize=8)

# 2. 遗忘率对比
forgetting_rates = [forgetting_baseline * 100, forgetting_ewc * 100, forgetting_replay * 100]
colors = ['lightcoral', 'lightyellow', 'lightgreen']
bars = axes[0, 1].bar(methods, forgetting_rates, color=colors, alpha=0.7)
axes[0, 1].set_ylabel('Forgetting Rate (%)')
axes[0, 1].set_title('Forgetting Comparison')
axes[0, 1].axhline(y=0, color='black', linestyle='--', linewidth=0.8)
axes[0, 1].grid(True, alpha=0.3, axis='y')

for bar, rate in zip(bars, forgetting_rates):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., rate,
                   f'{rate:.1f}%', ha='center', va='bottom' if rate > 0 else 'top', fontsize=10)

# 3. Replay 训练损失分解
axes[0, 2].plot(new_losses, label='Task B Loss', linewidth=2, color='skyblue')
axes[0, 2].plot(replay_losses, label='Replay Loss', linewidth=2, color='orange')
axes[0, 2].plot(losses, label='Total Loss', linewidth=2, color='purple', linestyle='--')
axes[0, 2].set_xlabel('Training Steps')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].set_title('Replay Training: Loss Decomposition')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. 方法特性雷达图
categories = ['No\nForgetting', 'Task B\nPerformance', 'Memory\nEfficiency', 
              'Privacy\nPreserving', 'Computation\nSpeed']

baseline_scores = [1, 5, 5, 5, 5]
ewc_scores = [3, 4, 5, 5, 3]
replay_scores = [4, 5, 2, 2, 4]

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
baseline_scores += baseline_scores[:1]
ewc_scores += ewc_scores[:1]
replay_scores += replay_scores[:1]
angles += angles[:1]

ax_radar = plt.subplot(2, 3, 4, projection='polar')
ax_radar.plot(angles, baseline_scores, 'o-', linewidth=2, label='Baseline', color='lightcoral')
ax_radar.fill(angles, baseline_scores, alpha=0.15, color='lightcoral')
ax_radar.plot(angles, ewc_scores, 's-', linewidth=2, label='EWC', color='lightyellow')
ax_radar.fill(angles, ewc_scores, alpha=0.15, color='lightyellow')
ax_radar.plot(angles, replay_scores, '^-', linewidth=2, label='Replay', color='lightgreen')
ax_radar.fill(angles, replay_scores, alpha=0.15, color='lightgreen')

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, size=9)
ax_radar.set_ylim(0, 5)
ax_radar.set_title('Method Characteristics', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax_radar.grid(True)

# 5. 方法对比表
comparison_data = [
    ['Metric', 'Baseline', 'EWC', 'Replay'],
    ['Task A (after)', f'{acc_A_after_baseline:.3f}', f'{acc_A_after_ewc:.3f}', f'{acc_A_after_replay:.3f}'],
    ['Task B', f'{acc_B_baseline:.3f}', f'{acc_B_ewc:.3f}', f'{acc_B_replay:.3f}'],
    ['Forgetting', f'{forgetting_baseline*100:.1f}%', f'{forgetting_ewc*100:.1f}%', f'{forgetting_replay*100:.1f}%'],
    ['Memory', 'Low', 'Low', 'Medium'],
    ['Privacy', 'Good', 'Good', 'Risk'],
]

axes[1, 1].axis('tight')
axes[1, 1].axis('off')
table = axes[1, 1].table(cellText=comparison_data, cellLoc='center', loc='center',
                         colWidths=[0.25, 0.25, 0.25, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# 设置表头样式
for i in range(4):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

axes[1, 1].set_title('Method Comparison Table', pad=20, fontsize=12, fontweight='bold')

# 6. 总结建议
axes[1, 2].axis('off')
recommendation_text = """
💡 Method Selection Guide

Use Baseline when:
  • No continual learning needed
  • Single task deployment

Use EWC when:
  • Privacy is critical
  • Cannot store old data
  • Memory constrained
  • Many sequential tasks

Use Experience Replay when:
  • Can store some data
  • Privacy not critical
  • Want best performance
  • Few sequential tasks

Best Practice:
  • Combine EWC + Replay
  • Use small replay buffer
  • Balance privacy & performance
"""

axes[1, 2].text(0.1, 0.5, recommendation_text, fontsize=10,
               verticalalignment='center', family='monospace',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("📊 Three Methods Comparison")
print("="*60)
print(f"\nBaseline (No Protection):")
print(f"  Task A: {acc_A_baseline:.3f} → {acc_A_after_baseline:.3f}")
print(f"  Forgetting: {forgetting_baseline*100:.1f}%")

print(f"\nEWC (Regularization):")
print(f"  Task A: {acc_A_ewc:.3f} → {acc_A_after_ewc:.3f}")
print(f"  Forgetting: {forgetting_ewc*100:.1f}%")

print(f"\nExperience Replay:")
print(f"  Task A: {acc_A_replay:.3f} → {acc_A_after_replay:.3f}")
print(f"  Forgetting: {forgetting_replay*100:.1f}%")

print(f"\n🏆 Best method: ", end="")
if forgetting_replay < forgetting_ewc:
    print("Experience Replay")
else:
    print("EWC")

print(f"\n💡 Key Takeaways:")
print(f"  • Both EWC and Replay significantly reduce forgetting")
print(f"  • Replay often performs better but needs memory")
print(f"  • EWC is privacy-preserving and memory-efficient")
print(f"  • Can combine both for best results!")

## 8. 实践指南与总结

### 8.1 何时使用哪种方法？

| 场景 | 推荐方法 | 原因 |
|------|----------|------|
| 有大量领域数据 | DAPT | 充分学习领域知识 |
| 数据有限但任务明确 | TAPT | 聚焦任务相关数据 |
| 需要保留旧任务 | EWC + Replay | 防止灾难性遗忘 |
| 多领域同时训练 | Adapters / MoE | 模块化，避免冲突 |
| 隐私敏感场景 | EWC | 无需存储旧数据 |
| 追求最佳性能 | DAPT + TAPT + Replay | 组合多种方法 |

### 8.2 最佳实践

**1. 数据准备**
- 收集高质量领域数据（至少 1M tokens for DAPT）
- 清洗和预处理（去除噪声）
- 评估领域偏移程度（词汇分析）

**2. 训练策略**
- 先 DAPT 后 TAPT（如果都适用）
- 使用较小学习率（1e-5 到 5e-5）
- 监控所有任务的性能（避免遗忘）
- 使用验证集早停

**3. 持续学习**
- 优先考虑 Experience Replay（如果可以存储数据）
- 隐私敏感时使用 EWC
- 多任务场景使用 Adapters
- 可以组合多种方法

**4. 评估**
- 测试所有领域/任务
- 计算平均性能和遗忘度
- 监控参数效率

### 8.3 常见问题 FAQ

**Q1: 需要多少领域数据？**

A: 取决于方法：
- **DAPT**: 至少 1M tokens，推荐 10M+ tokens
- **TAPT**: 10K-1M tokens 即可
- 数据质量比数量更重要

**Q2: 如何检测领域偏移？**

A: 多种方法：
- 比较词汇分布（词频、TF-IDF）
- 可视化嵌入空间（t-SNE, UMAP）
- 监控性能下降
- 计算分布距离（KL 散度、JS 散度）

**Q3: 能否适应多个领域？**

A: 可以，推荐方法：
- **Adapters**: 每个领域独立适配器
- **MoE**: 混合专家模型
- **Multi-task Learning**: 联合训练
- 注意平衡各领域数据

**Q4: EWC 和 Replay 哪个更好？**

A: 取决于场景：
- **Replay 更好**：性能通常更优
- **EWC 更好**：隐私敏感、内存受限
- **最佳**: 组合使用（小 buffer + EWC）

**Q5: 如何选择 EWC 的 λ 参数？**

A: 经验法则：
- 简单任务：λ = 1000-5000
- 复杂任务：λ = 10000-50000
- 通过验证集调优
- 观察旧任务性能和新任务学习速度的平衡

**Q6: 领域适应会损失通用能力吗？**

A: 可能会，但可以缓解：
- 使用较小学习率
- 不要过度训练
- 定期在通用数据上评估
- 考虑使用 Adapters（保留原模型）

### 8.4 性能提升预期

基于文献和实践经验：

| 方法 | 性能提升 | 训练成本 | 内存需求 |
|------|----------|----------|----------|
| 仅微调 | Baseline | 低 | 低 |
| DAPT | +2-3% | 中 | 低 |
| TAPT | +1-2% | 低 | 低 |
| DAPT + TAPT | +3-5% | 中 | 低 |
| EWC | -0.5-1% (trade-off) | 中 | 低 |
| Replay | -0-0.5% (trade-off) | 低 | 中 |
| Adapters | +1-3% | 低 | 低 |

### 8.5 工具和资源

**推荐库**：
- **Hugging Face Transformers**: 预训练模型
- **Hugging Face PEFT**: LoRA, Adapters
- **Avalanche**: 持续学习框架
- **PyTorch**: 深度学习框架

**数据集**：
- **医学**: PubMed, MIMIC-III
- **法律**: CaseHOLD, LegalBench
- **科学**: S2ORC, arXiv
- **代码**: CodeSearchNet, GitHub

**论文**：
- Don't Stop Pretraining (DAPT/TAPT)
- Overcoming Catastrophic Forgetting (EWC)
- Parameter-Efficient Transfer Learning (Adapters)

## 9. 本章总结

### 🎯 核心要点

1. **领域偏移是真实存在的问题**
   - 词汇分布、语义、句法都可能不同
   - 需要主动检测和应对

2. **持续预训练很有效**
   - DAPT: 大量领域数据，广泛提升
   - TAPT: 少量任务数据，聚焦提升
   - 组合使用效果最佳

3. **灾难性遗忘需要重视**
   - 顺序学习会导致严重遗忘
   - 必须使用防遗忘策略

4. **多种防遗忘方法**
   - EWC: 正则化，保护重要参数
   - Replay: 重放旧数据
   - Adapters: 架构隔离

5. **多领域学习是可行的**
   - Adapters 提供模块化解决方案
   - 避免冲突，易于扩展

### 📈 学习成果

完成本章后，你应该能够：

- ✅ 检测和量化领域偏移
- ✅ 实现完整的 DAPT/TAPT 流程
- ✅ 理解灾难性遗忘的原因和影响
- ✅ 实现 EWC 和 Experience Replay
- ✅ 设计多领域学习系统
- ✅ 在实际项目中选择合适的方法

### 🚀 下一步

**继续学习**：
- Module 6: 高级训练技术
- Module 7: 模型压缩与加速
- Module 8: 实际应用（RAG, Agent）

**实践项目**：
- 医学文本分类（DAPT + 微调）
- 多语言情感分析（Adapters）
- 持续学习聊天机器人（EWC + Replay）

### 💡 思考题

1. 为什么 TAPT 比 DAPT 数据需求更少但效果可能更好？
2. EWC 和 Experience Replay 各有什么优缺点？如何组合使用？
3. 如何设计一个能处理 10 个不同领域的模型？
4. 领域适应和迁移学习有什么区别和联系？
5. 在什么情况下领域适应可能不起作用或效果不佳？

---

**恭喜你完成 Module 5.3！** 🎉

你已经掌握了领域适应和持续学习的核心技术，这些是构建实用 AI 系统的关键能力。

In [ ]:
# 🔬 Micro Practice 7: 多领域学习（使用 Adapters）

"""
目标：实现基于 Adapter 的多领域学习系统
预期结果：理解如何同时处理多个领域
"""

class DomainAdapter(nn.Module):
    """领域特定的适配器"""
    def __init__(self, hidden_dim=20, adapter_dim=8):
        super().__init__()
        self.down_project = nn.Linear(hidden_dim, adapter_dim)
        self.up_project = nn.Linear(adapter_dim, hidden_dim)
        
    def forward(self, x):
        # Bottleneck architecture
        h = F.relu(self.down_project(x))
        output = self.up_project(h)
        return output

class MultiDomainModel(nn.Module):
    """支持多领域的模型"""
    def __init__(self, input_dim=10, hidden_dim=20, num_classes=2, num_domains=3):
        super().__init__()
        # 共享的主干网络
        self.shared_fc = nn.Linear(input_dim, hidden_dim)
        
        # 每个领域的适配器
        self.adapters = nn.ModuleList([
            DomainAdapter(hidden_dim, adapter_dim=8) 
            for _ in range(num_domains)
        ])
        
        # 分类头
        self.classifier = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x, domain_id):
        # 共享特征提取
        h = F.relu(self.shared_fc(x))
        
        # 领域特定适配
        adapter_output = self.adapters[domain_id](h)
        h = h + adapter_output  # Residual connection
        
        # 分类
        logits = self.classifier(h)
        return logits

def train_multi_domain(model, domains_data, epochs=100, lr=0.01):
    """多领域联合训练"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    history = {f'domain_{i}': [] for i in range(len(domains_data))}
    
    for epoch in range(epochs):
        epoch_losses = []
        
        # 对每个领域训练
        for domain_id, (X, y) in enumerate(domains_data):
            optimizer.zero_grad()
            
            # Forward pass
            logits = model(X, domain_id)
            loss = criterion(logits, y)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            epoch_losses.append(loss.item())
            history[f'domain_{domain_id}'].append(loss.item())
        
        if (epoch + 1) % 20 == 0:
            avg_loss = np.mean(epoch_losses)
            print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_loss:.4f}")
    
    return history

print("="*60)
print("Multi-Domain Learning with Adapters")
print("="*60)

# 生成三个领域的数据
print("\nGenerating data for 3 domains...")
X_domain0, y_domain0 = generate_task_data(0, n_samples=100)  # Domain A
X_domain1, y_domain1 = generate_task_data(1, n_samples=100)  # Domain B
X_domain2, y_domain2 = generate_task_data(2, n_samples=100)  # Domain C

domains_data = [
    (X_domain0, y_domain0),
    (X_domain1, y_domain1),
    (X_domain2, y_domain2)
]

print("✓ Data generated for 3 domains")

# 创建多领域模型
print("\nCreating multi-domain model with adapters...")
multi_model = MultiDomainModel(input_dim=10, hidden_dim=20, num_classes=2, num_domains=3)

# 计算参数量
total_params = sum(p.numel() for p in multi_model.parameters())
shared_params = sum(p.numel() for p in multi_model.shared_fc.parameters()) + \
                sum(p.numel() for p in multi_model.classifier.parameters())
adapter_params = sum(p.numel() for adapter in multi_model.adapters for p in adapter.parameters())

print(f"✓ Model created")
print(f"  Total parameters: {total_params}")
print(f"  Shared parameters: {shared_params}")
print(f"  Adapter parameters: {adapter_params} ({adapter_params/total_params*100:.1f}%)")

# 训练
print("\n" + "-"*60)
print("Training on all domains simultaneously...")
print("-"*60)
history = train_multi_domain(multi_model, domains_data, epochs=100, lr=0.01)

# 评估每个领域
print("\n" + "-"*60)
print("Evaluating on all domains...")
print("-"*60)
domain_accuracies = []
for domain_id, (X, y) in enumerate(domains_data):
    acc = evaluate(multi_model, X, y)
    domain_accuracies.append(acc)
    print(f"Domain {domain_id} accuracy: {acc:.3f}")

avg_accuracy = np.mean(domain_accuracies)
print(f"\nAverage accuracy across domains: {avg_accuracy:.3f}")

# 可视化
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. 训练曲线
ax1 = fig.add_subplot(gs[0, :2])
colors = ['skyblue', 'lightcoral', 'lightgreen']
for domain_id in range(3):
    losses = history[f'domain_{domain_id}']
    # 平滑曲线
    window = 10
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    ax1.plot(smoothed, label=f'Domain {domain_id}', color=colors[domain_id], linewidth=2)

ax1.set_xlabel('Training Steps')
ax1.set_ylabel('Loss')
ax1.set_title('Multi-Domain Training Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 领域性能雷达图
ax2 = fig.add_subplot(gs[0, 2], projection='polar')
categories = [f'Domain {i}' for i in range(3)]
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
values = domain_accuracies + domain_accuracies[:1]
angles += angles[:1]

ax2.plot(angles, values, 'o-', linewidth=2, color='purple', markersize=8)
ax2.fill(angles, values, alpha=0.25, color='purple')
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(categories)
ax2.set_ylim(0, 1)
ax2.set_title('Performance Across Domains', pad=20)
ax2.grid(True)

# 添加数值标签
for angle, value, cat in zip(angles[:-1], values[:-1], categories):
    ax2.text(angle, value + 0.05, f'{value:.2f}', ha='center', fontsize=10)

# 3. 领域性能对比
ax3 = fig.add_subplot(gs[1, 0])
domain_names = ['Domain A', 'Domain B', 'Domain C']
bars = ax3.bar(domain_names, domain_accuracies, color=colors, alpha=0.7)
ax3.set_ylabel('Accuracy')
ax3.set_title('Performance by Domain')
ax3.set_ylim([0, 1.1])
ax3.axhline(y=avg_accuracy, color='red', linestyle='--', label=f'Average: {avg_accuracy:.3f}')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, domain_accuracies):
    ax3.text(bar.get_x() + bar.get_width()/2., acc,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=10)

# 4. 参数分布
ax4 = fig.add_subplot(gs[1, 1])
param_categories = ['Shared\nBackbone', 'Adapters\n(3 domains)', 'Classifier']
param_counts = [
    sum(p.numel() for p in multi_model.shared_fc.parameters()),
    adapter_params,
    sum(p.numel() for p in multi_model.classifier.parameters())
]
colors_param = ['skyblue', 'lightcoral', 'lightgreen']
wedges, texts, autotexts = ax4.pie(param_counts, labels=param_categories, autopct='%1.1f%%',
                                     colors=colors_param, startangle=90)
ax4.set_title('Parameter Distribution')

# 5. Adapter 架构示意图
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
architecture_text = """
Multi-Domain Architecture

Input (10 dim)
    ↓
Shared FC (20 dim)
    ↓
┌───────┬───────┬───────┐
│Adapter│Adapter│Adapter│
│   A   │   B   │   C   │
└───────┴───────┴───────┘
    ↓
Classifier (2 classes)

Key Features:
• Shared backbone
• Domain-specific adapters
• Efficient: Only 30% params
• No catastrophic forgetting
• Easy to add new domains
"""

ax5.text(0.1, 0.5, architecture_text, fontsize=10,
        verticalalignment='center', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.suptitle('Multi-Domain Learning with Adapters', fontsize=14, fontweight='bold', y=0.98)
plt.show()

print("\n" + "="*60)
print("📊 Multi-Domain Learning Summary")
print("="*60)
print(f"\n✓ Successfully trained on 3 domains simultaneously")
print(f"  Average accuracy: {avg_accuracy:.3f}")
print(f"  Parameter efficiency: {adapter_params/total_params*100:.1f}% are adapters")
print(f"\n💡 Advantages of Adapter-based Multi-Domain Learning:")
print(f"  • No catastrophic forgetting (isolated adapters)")
print(f"  • Easy to add new domains (just add adapter)")
print(f"  • Parameter efficient (shared backbone)")
print(f"  • Modular and maintainable")
print(f"\n🎯 Use cases:")
print(f"  • Multi-lingual models")
print(f"  • Cross-domain text classification")
print(f"  • Personalized models for different users")

## 思考题参考答案

### 问题 1：为什么模型在新领域表现下降？领域偏移的本质是什么？

**领域偏移（Domain Shift）的定义：**

$$P_{\text{source}}(X, Y) \neq P_{\text{target}}(X, Y)$$

源域和目标域的联合分布不同，导致在源域训练的模型无法很好地泛化到目标域。

**性能下降的多层次原因：**

**1. 词汇分布差异（Lexical Shift）**

通用语言模型训练时，专业术语出现频率极低甚至未见过：
```
通用文本："心脏" 高频词
医学文本："左心室射血分数(LVEF)"、"心肌梗死(AMI)" 专业术语
```
模型对未见过的词汇无法生成有效的表示。

**2. 语义偏移（Semantic Shift）**

同一词汇在不同领域含义不同：

| 词汇 | 通用含义 | 医学含义 | 法律含义 |
|------|---------|---------|---------|
| "感染" | 传染疾病 | 病原体侵入宿主 | 受影响 |
| "批准" | 同意 | 临床试验通过 | 法院裁定 |
| "过敏" | 不喜欢 | 免疫系统过激反应 | — |

**3. 句法模式差异（Syntactic Shift）**

- 法律文本：极长句子、嵌套从句、被动语态密集
- 医学文本：缩写密集、列举结构、数字与单位混合
- 社交媒体：极短句、缩写、表情符号、语法不规则

**4. 任务分布差异（Label Shift）**

即使输入分布相同，标签分布也可能不同：
$$P_{\text{source}}(Y | X) \neq P_{\text{target}}(Y | X)$$

例如：通用情感分析模型将"副作用可以接受"判断为负面，但在医学语境中这是中性/正面评价。

**量化领域偏移的方法：**

1. **词汇重叠率（Vocabulary Overlap）**：目标域词汇中有多少在训练集中出现过
2. **JS 散度（Jensen-Shannon Divergence）**：衡量两个分布的差异程度
3. **困惑度差异（Perplexity Gap）**：语言模型在两个域上的困惑度差异

---

### 问题 2：DAPT 和 TAPT 各自解决什么问题？应该如何选择和组合？

**DAPT（Domain-Adaptive Pre-Training）：领域预训练**

- **目标**：解决词汇分布差异和领域语义偏移
- **方法**：在目标域的大量无标注文本上继续进行语言模型预训练
- **效果**：让模型"读懂"目标域的语言

$$\theta_{\text{DAPT}} = \arg\min_{\theta} \mathcal{L}_{\text{MLM/CLM}}(\mathcal{D}_{\text{domain}}, \theta_{\text{pretrained}})$$

**TAPT（Task-Adaptive Pre-Training）：任务预训练**

- **目标**：解决任务分布差异，让模型熟悉目标任务的数据模式
- **方法**：在任务训练集（不包含标签）上进行语言模型预训练
- **效果**：让模型"适应"目标任务的输入格式

$$\theta_{\text{TAPT}} = \arg\min_{\theta} \mathcal{L}_{\text{MLM/CLM}}(\mathcal{D}_{\text{task\_unlabeled}}, \theta)$$

**两者区别对比：**

| 维度 | DAPT | TAPT |
|------|------|------|
| 数据量 | 通常较大（GB级） | 较小（任务数据集本身）|
| 计算成本 | 高 | 低 |
| 解决的问题 | 领域词汇/语义差异 | 任务特定模式 |
| 效果 | 宏观领域适应 | 微观任务适应 |
| 依赖标注数据 | 否 | 否（只用文本） |

**选择与组合策略（来自 Gururangan et al. 2020）：**

```
最优组合：预训练 → DAPT → TAPT → 微调
性能提升（相对 BERT-base）：
- BERT-base only:          基线
- + DAPT:                  +1.5% F1
- + TAPT:                  +1.0% F1  
- + DAPT + TAPT:           +3.0% F1（最优）
```

**实际选择建议：**

- 计算资源充足：DAPT + TAPT 完整流程
- 资源有限：只做 TAPT（成本低，效果显著）
- 目标域数据极少无标注文本：只做 DAPT（利用更广泛的领域数据）

---

### 问题 3：什么是灾难性遗忘？为什么神经网络会出现这个问题？如何解决？

**灾难性遗忘（Catastrophic Forgetting）的定义：**

当神经网络学习新任务时，之前学习的旧任务性能急剧下降的现象。

$$\text{微调后}: \text{Task}_1 \text{ 性能} \downarrow\downarrow\downarrow, \quad \text{Task}_2 \text{ 性能} \uparrow$$

**产生原因的深层分析：**

1. **参数共享导致干扰**：神经网络使用同一组参数编码所有任务的知识，新任务的梯度更新会覆盖旧任务学到的权重方向。

2. **优化目标冲突**：
   $$\nabla_{\theta} \mathcal{L}_{\text{task2}} \cdot \nabla_{\theta} \mathcal{L}_{\text{task1}} < 0$$
   当两个任务的梯度方向相反时，优化新任务必然破坏旧任务。

3. **局部最优的破坏**：旧任务找到的参数局部最优点，对新任务而言并非最优，SGD 会将参数从旧任务的最优点推走。

**主要解决方案及原理：**

**方案 A：弹性权重巩固（EWC，Kirkpatrick et al. 2017）**

核心思想：识别对旧任务重要的参数，在学习新任务时对这些参数施加约束。

$$\mathcal{L}_{\text{EWC}} = \mathcal{L}_{\text{task2}}(\theta) + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta_{\text{task1}, i}^*)^2$$

其中 $F_i$（Fisher 信息矩阵对角线元素）衡量参数 $\theta_i$ 对旧任务的重要性。

| $F_i$ 大 | 参数对旧任务极重要，被大力"弹回"旧值 |
|----------|--------------------------------------|
| $F_i$ 小 | 参数对旧任务不重要，可自由更新 |

**方案 B：经验回放（Experience Replay）**

核心思想：在学习新任务时，混入少量旧任务的样本（记忆缓冲区），同时优化两个任务。

$$\mathcal{L}_{\text{replay}} = (1-\alpha)\mathcal{L}_{\text{task2}}(\theta) + \alpha \mathcal{L}_{\text{task1}}(\theta; \mathcal{M})$$

其中 $\mathcal{M}$ 是大小为 $M$ 的记忆缓冲区（通常每个任务保存 $M/T$ 个样本）。

**方案 C：进行性神经网络（Progressive Networks）**

为每个新任务添加新的网络列，旧网络列冻结，通过侧连接共享知识。

$$\text{优点：完全避免遗忘} \qquad \text{缺点：参数随任务数线性增长}$$

**方案 D：PEFT 方法（天然防遗忘）**

LoRA、Adapter 等方法冻结预训练权重，各任务只训练自己的增量参数，从根本上避免干扰。

**各方案对比：**

| 方法 | 防遗忘效果 | 计算成本 | 存储成本 | 推荐场景 |
|------|-----------|---------|---------|---------|
| EWC | ★★★★☆ | 低 | 低（Fisher矩阵）| 任务数少 |
| Experience Replay | ★★★★☆ | 中 | 中（缓冲区）| 通用场景 |
| Progressive Networks | ★★★★★ | 高 | 高 | 任务数极少 |
| PEFT（LoRA等）| ★★★★★ | 低 | 低 | 强烈推荐 |

---

### 问题 4：如何设计能同时处理多个领域的模型架构？

**多领域学习的核心挑战：**

1. **特征共享 vs. 特征专属**：哪些特征应该跨领域共享，哪些应该领域特有？
2. **负迁移（Negative Transfer）**：有时多任务学习反而降低某些任务的性能
3. **域标识可用性**：推理时是否知道输入属于哪个领域？

**架构设计方案：**

**方案 A：共享主干 + 领域专属头（最简单）**

```
输入
  ↓
共享 Transformer 编码器（冻结或低学习率）
  ↓
  ├── 领域A分类头 → A任务输出
  ├── 领域B分类头 → B任务输出
  └── 领域C分类头 → C任务输出
```

**方案 B：领域自适应层（Domain Adapter）**

在共享 Transformer 的每层中，为每个领域插入独立的 Adapter 模块：

$$h_{\text{domain}} = h_{\text{shared}} + \text{Adapter}_{\text{domain}}(h_{\text{shared}})$$

优点：共享通用知识 + 领域专属适配，参数高效

**方案 C：混合专家（Mixture of Experts, MoE）**

```python
class DomainMoE(nn.Module):
    def forward(self, x, domain_id=None):
        # 计算每个专家的权重
        gate_weights = self.gate(x)        # [batch, num_experts]
        expert_outputs = torch.stack([
            expert(x) for expert in self.experts
        ], dim=1)                           # [batch, num_experts, hidden]
        # 加权组合专家输出
        output = (gate_weights.unsqueeze(-1) * expert_outputs).sum(1)
        return output
```

**方案 D：领域提示（Domain Prompt）+ 统一模型**

使用领域特定的可学习提示向量，模型主体完全共享：

$$\text{输入} = [\underbrace{p_1, ..., p_k}_{\text{领域提示}}, x_1, ..., x_n]$$

不同领域对应不同的提示向量集合，推理时根据领域选择对应提示。

**实际工程推荐方案（按场景）：**

| 场景 | 推荐架构 | 原因 |
|------|---------|------|
| 领域差异小（如新闻+百科） | 共享主干 + 多头 | 简单高效 |
| 领域差异大（如医学+法律） | Domain Adapter | 平衡共享与专属 |
| 动态路由需求 | MoE | 自适应分配计算 |
| 资源极度受限 | Domain Prompt | 极少参数开销 |
| 领域标签未知 | 领域检测器 + 多路由 | 自动识别域 |

**关键设计原则：**

1. **底层共享，顶层专属**：底层特征（词法、句法）跨领域通用，顶层特征（语义、任务）领域专属
2. **渐进解耦**：先训练共享模型，再为各领域微调专属部分
3. **负迁移检测**：定期评估各领域性能，若某领域性能下降则增加其专属参数比例
4. **领域均衡采样**：训练时按领域大小进行逆比例采样，防止大领域"压制"小领域

**完整多领域系统架构：**

```
输入文本
    ↓
[领域检测器] → 领域ID
    ↓
共享预训练编码器（LoRA全局适配）
    + 领域专属 Adapter
    ↓
任务输出头（每领域独立）
    ↓
最终预测
```